# RL-Trained Tool-Use Agent

Train Qwen 2.5 0.5B to call tools via GRPO using OpenPipe/ART.

**Runtime:** A100 GPU (Colab Pro)  
**Estimated cost:** ~$10-15 (within Pro's 100 monthly compute units)  
**Estimated time:** ~3 hours

## Before you start

1. **Runtime → Change runtime type → A100 GPU** (Colab Pro required)
2. Add two secrets via the 🔑 key icon in the left sidebar:
   - `GITHUB_TOKEN` — a fine-grained GitHub PAT with read access to `abhid1234/RL-Trained-Tool-Use-Agent`
   - `HF_TOKEN` — a HuggingFace write token
3. Toggle both secrets to **Notebook access**
4. Run cells top-to-bottom

**If you walk away for >15 min:** Runtime → Disconnect runtime (saves idle burn). Reconnect when you're back.

**If anything breaks:** see RUNBOOK.md for known issues + fixes, or RUNPOD.md if you're running on RunPod instead (which may be more reliable for this specific stack).

**Prefer RunPod?** If Colab dependency conflicts bite again, see RUNPOD.md — same project runs on a RunPod PyTorch 2.8 pod in ~20 min of setup.

In [ ]:
# Cell 1: Verify runtime is A100
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to A100 GPU'

In [ ]:
# Cell 2: Install ART with the [backend] extra.
#
# ROOT CAUSE (learned 2026-04-12 after ~7 hours of dep hell):
# ART uses a CUSTOM vLLM fork distributed as a direct wheel URL from GitHub,
# NOT upstream vLLM from PyPI. Every attempt to pin upstream vLLM (0.7, 0.10,
# 0.11, 0.16) fails because ART's patches target APIs that only exist in the
# custom fork `vllm-0.17.0+art1`.
#
# The fix: install `openpipe-art[backend]==0.5.17`. The [backend] extra pulls
# the custom vLLM wheel alongside exact pins for trl==0.20.0, transformers==5.2.0,
# unsloth==2026.3.3, and everything else in ART's tested stack.
#
# IMPORTANT: The backend stack requires torch >= 2.8.0. Colab's A100 runtime
# typically ships with torch 2.4-2.6. Pip install will attempt to upgrade
# torch in-place — this occasionally succeeds, but it's fragile. If it fails,
# switch to RunPod with the runpod/pytorch:2.8.0-py3.11-cuda12.8.1 template.
# See RUNPOD.md.

!pip install --quiet uv
!uv pip install --system 'openpipe-art[backend]==0.5.17' 'pyyaml>=6.0' 'huggingface_hub'

# Verify the install: vllm MUST print `0.17.0+art1` (the custom fork).
# If it prints any other vllm version, the install pulled upstream vLLM by
# mistake — re-run Cell 2 or fall back to RunPod.
!python3 -c "import torch, art, vllm, trl, unsloth; print(f'torch={torch.__version__}, vllm={vllm.__version__}, trl={trl.__version__}, unsloth={unsloth.__version__}, all imports OK')"

In [ ]:
# Cell 3: Clone the private repo using a Colab secret
from google.colab import userdata
import subprocess

gh_token = userdata.get('GITHUB_TOKEN')
repo_url = f'https://{gh_token}@github.com/abhid1234/RL-Trained-Tool-Use-Agent.git'

!rm -rf /content/RL-Trained-Tool-Use-Agent
subprocess.run(['git', 'clone', repo_url, '/content/RL-Trained-Tool-Use-Agent'], check=True)
%cd /content/RL-Trained-Tool-Use-Agent
!git log --oneline -5

In [ ]:
# Cell 4: Run test suite to verify clean checkout
!python3 -m pytest tests/ -v

In [ ]:
# Cell 5: Regenerate scenarios (reproducible from seed)
!python3 scripts/generate_scenarios.py
!wc -l data/scenarios_train.jsonl data/scenarios_val.jsonl

In [ ]:
# Cell 6: Smoke test — 2 steps × 32 rollouts (~3 min).
#
# Why 32 rollouts and not 8?
# GRPO computes advantages across groups of N scenarios × M generations.
# 32 rollouts with num_generations=8 → 4 groups per step. ART needs
# multiple groups to compute cross-group advantages. With only 1 group
# (rollouts=8), training gets skipped even if there's within-group variance.
#
# `rm -rf .art` clears stale checkpoint state from previous runs — without
# this, a rerun may see `start_step=N` and exit immediately if num_steps <= N.
#
# What to look for in the output:
#  ✅ `GRPO config: 4 scenarios/step × 8 rollouts/scenario = 32 rollouts/step`
#  ✅ vLLM engine init mentions `0.17.0+art1` somewhere
#  ✅ 4 per-group variance lines, at least one with std > 0
#  ✅ A `loss=X.XXX` or similar non-SKIP message in the ART output
#  ❌ If you see `SKIP: all rewards equal` — check that src/rollout.py uses
#     `logprobs=True` and stores raw Choice objects (not dict conversions)
!rm -rf .art
!python3 -m src.train --steps 2 --rollouts 32

In [ ]:
# Cell 7: Full GRPO training — 50 steps × 96 rollouts (~3 hours).
#
# Only run this after Cell 6 shows training actually happening (not SKIP).
#
# With num_generations=8 and rollouts_per_step=96, each step trains on
# 12 scenarios × 8 attempts each = 96 rollouts. Over 50 steps, that's
# 600 scenario-instances of training signal.
#
# Watch the 'Avg reward' line climb step by step. Healthy training:
#   step 1-5:   avg reward ~-1.5 to -0.5 (model exploring, mostly wrong)
#   step 10-20: avg reward ~+0.5 to +1.5 (starting to find patterns)
#   step 30-50: avg reward ~+2.5 to +3.5 (nailing multi-step chains)
#
# If reward stays flat at -3 for 10+ steps, stop and check src/rollout.py.
# If reward climbs then crashes, lower learning rate (--lr 2e-5).
!rm -rf .art
!python3 -m src.train --config configs/train_config.yaml

In [ ]:
# Cell 8: Evaluate RL-trained model on the validation set.
!python3 -m src.evaluate --scenarios data/scenarios_val.jsonl

In [ ]:
# Cell 9: Push the trained model to HuggingFace Hub
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

HF_USERNAME = 'abhidaas'  # Change to your HuggingFace username

!python3 scripts/export_model.py \
    --checkpoint-dir ./checkpoints/qwen-0.5b-tool-agent \
    --repo-id {HF_USERNAME}/qwen-0.5b-tool-agent-grpo \
    --merge-adapter